<a href="https://colab.research.google.com/github/Lingadarini/trendpulse/blob/part2-numpy/part2_numpy_pandas/part2_numpy_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!git clone -b part2-numpy https://github.com/Lingadarini/trendpulse.git
!ls trendpulse/part2_numpy_pandas/data

Cloning into 'trendpulse'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 60 (delta 21), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 2.05 MiB | 4.76 MiB/s, done.
Resolving deltas: 100% (21/21), done.
categories.csv	GBvideos.csv  INvideos.csv  USvideos.csv


# Part 2 - NumPy and Pandas

In [5]:
import numpy as np
import pandas as pd

DATA_DIR = "trendpulse/part2_numpy_pandas/data/"
COUNTRIES = ["US", "GB", "IN"]

frames = {}
for country in COUNTRIES:
    df_c = pd.read_csv(f"{DATA_DIR}{country}videos.csv")
    df_c["country"] = country
    frames[country] = df_c
    print(f"{country}: shape = {df_c.shape}")

combined = pd.concat(frames.values(), ignore_index=True)
print("Combined shape:", combined.shape)
print()
print(combined[["views", "likes", "dislikes", "comment_count"]].dtypes)

US: shape = (16080, 17)
GB: shape = (15075, 17)
IN: shape = (15075, 17)
Combined shape: (46230, 17)

views              int64
likes            float64
dislikes           int64
comment_count    float64
dtype: object


## NumPy arrays

In [6]:
# Fill blanks with 0 first, so the arrays have no NaN values
views = combined["views"].to_numpy()
likes = combined["likes"].fillna(0).to_numpy()
dislikes = combined["dislikes"].to_numpy()
comments = combined["comment_count"].fillna(0).to_numpy()

for name, arr in [("views", views), ("likes", likes),
                  ("dislikes", dislikes), ("comments", comments)]:
    print(f"{name:9s} shape={arr.shape}  dtype={arr.dtype}")

# Indexing and slicing
top10 = np.sort(views)[::-1][:10]
print("\n10 largest view counts:", top10)

print("Every 100th record (first 5):", views[::100][:5])
print("Number of every-100th records:", len(views[::100]))

sub = views[1000:1010]
print("Sub-array views[1000:1010]:", sub)

views     shape=(46230,)  dtype=int64
likes     shape=(46230,)  dtype=float64
dislikes  shape=(46230,)  dtype=int64
comments  shape=(46230,)  dtype=float64

10 largest view counts: [360760164 343297002 286714357 282812154 255362869 236275528 230704137
 210747294 200444300 181785943]
Every 100th record (first 5): [ 48555  21869 245258 421352  54521]
Number of every-100th records: 463
Sub-array views[1000:1010]: [ 414515   88881  155347  631406 1261551  798063  930369  341965  277706
 2943874]


## Engagement score (NumPy only)

In [7]:
# (a) Safe ratios: np.where avoids dividing by zero
votes = likes + dislikes
like_ratio = np.where(votes > 0, likes / np.where(votes > 0, votes, 1), 0.0)
comment_rate = np.where(views > 0, comments / np.where(views > 0, views, 1), 0.0)

# (b) Min-max normalisation: scales each array to the range 0..1
def min_max(arr):
    span = arr.max() - arr.min()
    return (arr - arr.min()) / span if span > 0 else np.zeros_like(arr, dtype=float)

views_norm = min_max(views.astype(float))
likes_norm = min_max(likes)
comments_norm = min_max(comments)

print("like_ratio   : min", like_ratio.min().round(3), " max", like_ratio.max().round(3))
print("comment_rate : min", comment_rate.min().round(5), " max", comment_rate.max().round(5))
print("views_norm   : min", views_norm.min(), " max", views_norm.max())
print("likes_norm   : min", likes_norm.min(), " max", likes_norm.max())
print("comments_norm: min", comments_norm.min(), " max", comments_norm.max())

like_ratio   : min 0.0  max 1.0
comment_rate : min 0.0  max 0.02463
views_norm   : min 0.0  max 1.0
likes_norm   : min 0.0  max 1.0
comments_norm: min 0.0  max 1.0


In [8]:
# (c) Weighted engagement score
W_VIEWS, W_LIKES, W_COMMENTS = 0.2, 0.4, 0.4
engagement = (W_VIEWS * views_norm
              + W_LIKES * likes_norm
              + W_COMMENTS * comments_norm)

# (d) Log-scaled views (log1p handles zeros safely)
log_views = np.log1p(views)

# Boolean mask: top 10% by engagement
cutoff = np.percentile(engagement, 90)
top10_mask = engagement >= cutoff

print("Weights:", W_VIEWS, W_LIKES, W_COMMENTS, "(sum =", W_VIEWS + W_LIKES + W_COMMENTS, ")")
print("Engagement: min", engagement.min().round(4), " max", engagement.max().round(4))
print("90th percentile cutoff:", cutoff.round(5))
print("Records flagged top 10%:", top10_mask.sum(), "of", len(engagement))
print("Raw views range     :", f"{views.min():,} to {views.max():,}")
print("Log views range     :", log_views.min().round(2), "to", log_views.max().round(2))

Weights: 0.2 0.4 0.4 (sum = 1.0 )
Engagement: min 0.0  max 0.9076
90th percentile cutoff: 0.00567
Records flagged top 10%: 4623 of 46230
Raw views range     : 1,066 to 360,760,164
Log views range     : 6.97 to 19.7


### Why these weights

| Component | Weight | Reason |
|---|---|---|
| Views (normalised) | 0.2 | Views show reach, but a video can be watched by many people who do not care about it, so views get the smallest weight |
| Likes (normalised) | 0.4 | A like is an active choice by a viewer, so it shows approval |
| Comments (normalised) | 0.4 | A comment takes the most effort, so it is the strongest sign of real engagement |

The weights add up to 1, so the score stays between 0 and 1.

### Why raw views are hard to plot
Views are extremely skewed: most videos have a few hundred thousand views, but a handful have hundreds of millions. On a normal axis these few huge values stretch the scale and squash almost every other video into one corner of the chart. Taking the logarithm (log1p) compresses the range so the full spread of videos becomes visible.

## Reshape and broadcasting

In [9]:
# Convert the trending_date strings (yy.dd.mm) to real dates
combined["trending_dt"] = pd.to_datetime(combined["trending_date"], format="%y.%d.%m")

# (days x countries) table of daily trending counts, as a NumPy array
counts_table = pd.crosstab(combined["trending_dt"], combined["country"])[COUNTRIES]
counts = counts_table.to_numpy()
print("counts shape (days x countries):", counts.shape)
print("First 3 days:\n", counts[:3])

# reshape and flatten
flat = counts.flatten()
print("\nflatten shape:", flat.shape)
back = flat.reshape(counts.shape)
print("reshape back equals original:", np.array_equal(back, counts))

# Broadcasting 1: each country's share of each day's total
day_totals = counts.sum(axis=1, keepdims=True)           # shape (days, 1)
share = counts / np.where(day_totals == 0, 1, day_totals)
print("\nShares of day 1:", share[0].round(3), "(sum =", share[0].sum().round(3), ")")

# Broadcasting 2: each day's value relative to that country's average
country_avg = counts.mean(axis=0)                        # shape (countries,)
relative = counts / country_avg
print("Country averages:", country_avg.round(1))
print("Day 1 relative to average:", relative[0].round(2))

counts shape (days x countries): (80, 3)
First 3 days:
 [[200 200 200]
 [202 200 201]
 [200 200 200]]

flatten shape: (240,)
reshape back equals original: True

Shares of day 1: [0.333 0.333 0.333] (sum = 1.0 )
Country averages: [201.  188.4 188.4]
Day 1 relative to average: [1.   1.06 1.06]


## Pandas: loading and inspection

In [10]:
for name, frame in list(frames.items()) + [("COMBINED", combined)]:
    print("=" * 60)
    print(name, "shape:", frame.shape)
    print("-" * 60)
    frame.info()
    print()

combined.head()

US shape: (16080, 17)
------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16080 entries, 0 to 16079
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   video_id                16080 non-null  object 
 1   trending_date           16080 non-null  object 
 2   title                   16080 non-null  object 
 3   channel_title           16080 non-null  object 
 4   category_id             16080 non-null  int64  
 5   publish_time            16080 non-null  object 
 6   tags                    16080 non-null  object 
 7   views                   16080 non-null  int64  
 8   likes                   16076 non-null  float64
 9   dislikes                16080 non-null  int64  
 10  comment_count           16068 non-null  float64
 11  thumbnail_link          16080 non-null  object 
 12  comments_disabled       16080 non-null  bool   
 13  ratings_

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country,trending_dt
0,g_N-w8LeAi8,17.14.11,Honest Football Prank LEAKED,QuietSpeech10,22,2017-11-14T07:04:28.000Z,Monsoon|Robots,48555,2189.0,180,136.0,https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US,2017-11-14
1,Ugyql2NlPAd,17.14.11,Surprising Cricket Review,IncredibleChallenge41,25,2017-11-12T22:35:48.000Z,Winter,198385,1159.0,492,1446.0,https://i.ytimg.com/vi/Ugyql2NlPAd/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US,2017-11-14
2,8fE19jkxJ9n,17.14.11,Mira Solis - Football (Live) OFFICIAL,WildTutorial42,10,2017-11-13T08:40:22.000Z,Cricket,231656,8742.0,896,432.0,https://i.ytimg.com/vi/8fE19jkxJ9n/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US,2017-11-14
3,WOY0nn7HCsO,17.14.11,Zed Rivera - Space (Lyric Video)?!,ShockingInterview30,10,2017-10-27T22:21:47.000Z,Festival|Guitar,8083,176.0,8,19.0,https://i.ytimg.com/vi/WOY0nn7HCsO/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US,2017-11-14
4,2Lc_Cs5VVr_,17.14.11,Surprising Election Recipe,UltimateVlog96,27,2017-11-09T13:20:20.000Z,Festival,373161,11305.0,167,891.0,https://i.ytimg.com/vi/2Lc_Cs5VVr_/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US,2017-11-14


In [11]:
combined[["views", "likes", "dislikes", "comment_count"]].describe()

,views,likes,dislikes,comment_count
count,4.623000e+04,4.621800e+04,46230.000000,46194.000000
mean,1.178603e+06,4.580005e+04,2325.983625,3449.773347
std,5.916188e+06,2.969554e+05,10518.962756,13665.153495
min,1.066000e+03,0.000000e+00,0.000000,0.000000
25%,1.294632e+05,3.656000e+03,213.000000,404.000000
50%,3.182775e+05,1.020100e+04,594.000000,1074.000000
75%,8.240505e+05,2.898775e+04,1666.000000,2779.000000
max,3.607602e+08,2.753782e+07,618175.000000,778390.000000


In [12]:
# Convert date columns to real datetimes
combined["trending_dt"] = pd.to_datetime(combined["trending_date"], format="%y.%d.%m")
combined["publish_dt"] = pd.to_datetime(combined["publish_time"])

# Data-quality checks
print("1. Missing values per column:")
print(combined.isna().sum()[combined.isna().sum() > 0])

print("\n2. Duplicate rows (same video_id + trending_date + country):",
      combined.duplicated(subset=["video_id", "trending_date", "country"]).sum())

print("\n3. Fully identical rows:", combined.duplicated().sum())

print("\n4. Removed/errored videos:", combined["video_error_or_removed"].sum())
print("5. Comments disabled      :", combined["comments_disabled"].sum())
print("6. Ratings disabled       :", combined["ratings_disabled"].sum())

print("\n7. Videos published AFTER they trended:",
      (combined["publish_dt"].dt.tz_localize(None) > combined["trending_dt"]).sum())

print("\nDate types now:")
print(combined[["trending_dt", "publish_dt"]].dtypes)

1. Missing values per column:
likes              12
comment_count      36
description      1624
dtype: int64

2. Duplicate rows (same video_id + trending_date + country): 230

3. Fully identical rows: 230

4. Removed/errored videos: 92
5. Comments disabled      : 1048
6. Ratings disabled       : 544

7. Videos published AFTER they trended: 6164

Date types now:
trending_dt         datetime64[ns]
publish_dt     datetime64[ns, UTC]
dtype: object


In [13]:
pub_date = combined["publish_dt"].dt.tz_localize(None).dt.normalize()   # drop the time of day
trend_date = combined["trending_dt"]

after = pub_date > trend_date
same_day = pub_date == trend_date

print("Published on a LATER date than trending:", after.sum())
print("Published on the SAME date as trending :", same_day.sum())
print("Published on an EARLIER date           :", (pub_date < trend_date).sum())

print()
print(combined.loc[after, ["video_id", "country", "trending_dt", "publish_dt"]].head(5))

Published on a LATER date than trending: 0
Published on the SAME date as trending : 6164
Published on an EARLIER date           : 40066

Empty DataFrame
Columns: [video_id, country, trending_dt, publish_dt]
Index: []


In [14]:
before = len(combined)

# What counts as a duplicate: same video_id on the same trending_date in the same country
clean = combined.drop_duplicates(subset=["video_id", "trending_date", "country"]).copy()
print(f"Rows before: {before:,}   after: {len(clean):,}   duplicates removed: {before - len(clean)}")

# Missing values: count, percentage, strategy
missing = clean.isna().sum()
missing = missing[missing > 0]
for col, n in missing.items():
    print(f"{col}: {n} missing ({n / len(clean) * 100:.2f}%)")

clean["likes"] = clean["likes"].fillna(0).astype(int)
clean["comment_count"] = clean["comment_count"].fillna(0).astype(int)
clean["description"] = clean["description"].fillna("")

print("\nMissing values after cleaning:", clean.isna().sum().sum())
print("Shape now:", clean.shape)

Rows before: 46,230   after: 46,000   duplicates removed: 230
likes: 12 missing (0.03%)
comment_count: 36 missing (0.08%)
description: 1615 missing (3.51%)

Missing values after cleaning: 0
Shape now: (46000, 19)


## Data-quality issues and decisions

| # | Issue found | Count | How it was handled |
|---|---|---|---|
| 1 | Duplicate rows (same video_id, same trending_date, same country) | 230 | Removed with drop_duplicates, keeping the first. A video can trend on many days and in many countries, so only the same video on the same day in the same country counts as a duplicate |
| 2 | Missing likes | 12 (0.03%) | Filled with 0. The number is tiny and a blank most likely means none were recorded |
| 3 | Missing comment_count | 36 (0.08%) | Filled with 0, same reason |
| 4 | Missing description | 1,615 (3.51%) | Filled with an empty string. The column is not used in the analysis |
| 5 | Removed or errored videos (video_error_or_removed = True) | 92 | Kept and flagged. They are few, and their view counts are still valid records |
| 6 | Comments disabled / ratings disabled | 1,048 / 544 | Kept. The like and comment numbers for these are 0 because the feature was off, not because nobody engaged |
| 7 | Dates stored as text (trending_date as yy.dd.mm, publish_time as ISO text) | all rows | Converted to real datetimes with pd.to_datetime |
| 8 | likes and comment_count stored as floats because of the blanks | all rows | Converted to integers after filling the blanks |
| 9 | Apparent "published after trending" records | 6,164 | Checked: every one was published on the same calendar day as it trended, only later than midnight. No date inconsistency, nothing removed |